# 22 · Thiên lệch "có" của SH — chẩn đoán logit + ngưỡng lúc suy luận (W13, R-YB **v3**)

**Vấn đề (W12, POPE-vi):** adapter dùng chung **SH** (adapter của prototype) trả lời "có" ở 77–89% câu "Trong ảnh có X không?". So với zero-shot B1-ocr, ΔF1 là −5,7 / −9,2 / −9,4. Câu hỏi: đây là **lệch ngưỡng** (dời ngưỡng là sửa được) hay **mất khả năng phân biệt** (phải train lại)?

**Lần chạy v1 (27/10) = KHÔNG HỢP LỆ**: quyết định logit chỉ trùng greedy 87,4% (SH) / 94,4% (B1-ocr), dưới ngưỡng 98%. Chẩn đoán (`experiments/W13_yesno_calib_L4_seed42/v1_diagnosis.md`):
1. Từ "không" viết thường là **2 token** (`kh` + `ông`), "Có" là `C` + `ó` → v1 bỏ sót vì chỉ lấy từ 1 token.
2. `generate()` áp **repetition penalty** (từ `generation_config`) lên token đã có trong prompt → greedy bị phạt chữ "có". v1 không áp penalty nên không tái hiện được greedy.

**Lần chạy v2 (27/10) = dừng ở smoke test** (trùng greedy 96,7% < 98%): v2 **cộng dồn** xác suất mỗi phía, còn greedy chọn **token tốt nhất**; 8/10 câu lệch là đúng kiểu này, 2/10 là gần hoà do chạy lô 2 hàng.

**Cách đo v3** (ngưỡng luật giữ nguyên, ghi trước ở `docs/W13/README.md`):
- **Một forward 1 hàng** — y hệt bước đầu của `generate()` (cùng cách gọi model như v1, đã chạy trơn 9 run).
- Áp **đúng repetition penalty như `generate()`** (`generation_config` của Qwen2.5-VL-3B-Instruct: **1,05**).
- `margin = max log p(token đầu phía có: có, C, yes, Yes) − max log p(token đầu phía không: kh, Không, no, No)` — **so token tốt nhất mỗi phía như greedy**; τ = 0 ⇔ greedy; quyết định "có" ⇔ `margin > τ`; τ chọn **chỉ trên calibration**.
- Selftest so với **chính `model.generate()`** trên Qwen2.5-VL tí hon kiến trúc thật: điểm trùng tuyệt đối.

**Luật R-YB** — ĐẠT khi đủ cả: (1) trùng greedy W12 ≥ 98% (SH, B1-ocr); (2) cận dưới CI95 ΔF1 so với B1-ocr greedy > −2 ở cả 3 cách chọn; (3) yes-ratio ≤ 60% ở cả 3 cách chọn.

| Mục | Runtime | Việc | Thời gian |
|---|---|---|---|
| 1–2 | L4 | thiết lập, kiểm adapter + OCR + 2 file câu hỏi, selftest | ≈ 10–15 phút (có chép ảnh) |
| 3 | L4 | **smoke test** 300 câu SH: trùng greedy ≥ 98% mới chạy tiếp | ≈ 2 phút |
| 4 | L4 | 3 run **bắt buộc** (SH calib, SH POPE-vi, B1-ocr POPE-vi), rồi 6 run chỉ mô tả | ≈ 1 h + ≈ 2 h |
| 5–6 | L4 | phân tích + phán quyết, tự trả máy | ≈ 1 phút |

## 1. Thiết lập môi trường

Mount Drive, lấy code mới nhất từ GitHub, khai báo đường dẫn. Notebook chạy **toàn bộ trên L4**.

*Kỳ vọng: commit mới nhất (`W13 step 1d …`), `Runtime: NVIDIA L4`, `✅ Thiết lập xong`.*

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, json, hashlib, subprocess
if not os.path.exists('/content/ViVQA-VLM/.git'):
    !cd /content && git clone -q https://github.com/PLHGNAOH/ViVQA-VLM.git
%cd /content/ViVQA-VLM
!git pull -q origin main
!git log -1 --oneline
sys.path.insert(0, '/content/ViVQA-VLM')
from src.eval.yesno_calib import SCORING_VERSION
assert SCORING_VERSION == 'first-token-max-reppen-v3', 'Code cũ -> git pull chưa lấy được commit R-YB v3'

DRIVE_ROOT = '/content/drive/MyDrive/ViVQA-VLM'
EXP_ROOT = f'{DRIVE_ROOT}/experiments'
POPE_REPO = 'experiments/W12_pope_vi_questions/pope_vi_questions_v1.json'   # bộ câu hỏi W12 đã chốt
POPE_SHA = '53d04eb23ad41ecee372d6dee01201578de75d992d33dd2c5b94d4aa328928fd'
Q_DIR = 'experiments/W13_yesno_questions'                                    # calib + t2, đã commit (sinh ở v1)
OUT = f'{EXP_ROOT}/W13_yesno_calib_v3_L4_seed42'
CHUNK = 500

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(1 << 20), b''):
            h.update(b)
    return h.hexdigest()

def run_cmd(cmd, check=True):
    # Chạy lệnh, in log trực tiếp; lỗi -> DỪNG notebook (trừ khi check=False: trả mã lỗi)
    print('$', cmd)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='')
    rc = p.wait()
    if check and rc != 0:
        raise RuntimeError(f'Lệnh lỗi (mã {rc}) -> xem log ở trên, gửi Claude')
    return rc

GPU = subprocess.run('nvidia-smi --query-gpu=name --format=csv,noheader', shell=True,
                     capture_output=True, text=True).stdout.strip() or 'CPU'
print('Runtime:', GPU)
print('✅ Thiết lập xong')

## 2. Kiểm tra trước khi tốn GPU

Phần này **bắt buộc L4**, cùng GPU với các run greedy W12. Nếu đang ở runtime khác thì vào **Runtime → Change runtime type → L4 → Save**, rồi **Runtime → Run all**.

Các bước kiểm:
- **(a)** Cài thư viện, cùng pin `transformers==5.17.0`.
- **(b)** Cấu hình OCR đọc từ `ocr_selection.json` (W08). OCR cache ViVQA phải trùng sha256 biên bản W11.
- **(c)** Adapter SH và B3-full trên Drive phải trùng dấu vân tay trọng số với run gốc.
- **(d)** POPE-vi = bản W12. Hai file calib + t2 trong repo = bản đã chốt (sinh ở v1), và bản trên Drive phải trùng byte.
- **(e)** Selftest `yesno_calib` (phần toán + so với chính `model.generate()` trên Qwen2.5-VL tí hon) + `pope_vi`.

*Kỳ vọng: `GPU: NVIDIA L4`, `paddledet_vietocr | v1 | 600 | 0.0`, `OCR cache ViVQA = biên bản W11 ✅`, 2 dòng `adapter … = run gốc ✅`, `câu hỏi: 3 file = bản chốt ✅`, `✅ selftest OK (2 script)`.*

In [ ]:
if 'L4' not in GPU:
    raise SystemExit('❌ Notebook 22 phải chạy trên L4 (cùng GPU với run greedy W12). '
                     'Runtime -> Change runtime type -> L4, rồi Run all.')
print('GPU:', GPU)
!pip install -q transformers==5.17.0 accelerate qwen-vl-utils bitsandbytes peft huggingface_hub pyyaml

from src.data.ocr import default_cache_path, load_ocr_cache
from src.eval.run_baseline import adapter_fingerprint
from src.eval.pope_vi import REF_EVAL
from src.eval.yesno_calib import QFILES

# (b) OCR: cấu hình đóng băng W08 + cache ViVQA đúng biên bản W11
sel = json.load(open('experiments/W08_ocr_select_dev300_L4_seed42/ocr_selection.json', encoding='utf-8'))['decision']
BACKEND, TPL, MAXC, MINC = sel['backend'], sel['template'], sel['max_chars'], sel['min_conf']
print(BACKEND, '|', TPL, '|', MAXC, '|', MINC)
OCR_ARGS = f'--mode ocr --ocr_backend {BACKEND} --ocr_template {TPL} --ocr_max_chars {MAXC} --ocr_min_conf {MINC}'
C_VV = default_cache_path(DRIVE_ROOT, 'vivqa', BACKEND)
MAN_VV = f'{EXP_ROOT}/W11_ocr_cache_vivqa/ocr_cache_manifest.json'
assert sha256_file(C_VV) == json.load(open(MAN_VV, encoding='utf-8'))['sha256'], 'OCR cache ViVQA KHÁC biên bản W11 -> dừng, báo Claude'
print('OCR cache ViVQA = biên bản W11 ✅')

# (c) adapter = đúng adapter của run gốc (so dấu vân tay trọng số, không so đường dẫn)
AD = {}
for k in ('SH', 'B3full'):
    ref = json.load(open(f'{EXP_ROOT}/{REF_EVAL[k]}/metrics.json', encoding='utf-8'))['meta']['adapter']
    fp = adapter_fingerprint(ref['path'])
    assert fp['weights_sha256'] == ref['weights_sha256'], f'{k}: adapter trên Drive KHÁC run gốc -> dừng, báo Claude'
    AD[k] = ref['path']
    print(f"adapter {k:6s} = run gốc ✅  ({fp['weights_mb']} MB, {fp['weights_sha256'][:12]}…)  {ref['path']}")

# (d) câu hỏi: dùng bản trong repo (đã commit), bản Drive (sinh ở v1) phải trùng byte
assert sha256_file(POPE_REPO) == POPE_SHA, 'POPE-vi trong repo KHÁC bản W12 -> dừng, báo Claude'
Q_FILES = {'calib': f"{Q_DIR}/{QFILES['calib']}", 'popevi': POPE_REPO, 'popevi_t2': f"{Q_DIR}/{QFILES['popevi_t2']}"}
for k in ('calib', 'popevi_t2'):
    drive_copy = f"{EXP_ROOT}/W13_yesno_questions/{QFILES[k]}"
    assert sha256_file(Q_FILES[k]) == sha256_file(drive_copy), f'{k}: bản repo KHÁC bản Drive -> dừng, báo Claude'
cal = json.load(open(Q_FILES['calib'], encoding='utf-8'))['data']
need, have = {q['image'] for q in cal}, set(load_ocr_cache(C_VV)['images'])
assert need <= have, 'OCR cache thiếu ảnh calibration -> báo Claude'
for k, p in Q_FILES.items():
    print(f'{k:10s} {len(json.load(open(p, encoding="utf-8"))["data"]):5d} câu | sha256 {sha256_file(p)[:12]}…')
print('câu hỏi: 3 file = bản chốt ✅')

# (e) selftest (Colab có torch -> chạy cả phần toán)
for mod, must in {'src.eval.yesno_calib': 'yesno_calib selftest OK', 'src.eval.pope_vi': 'pope_vi selftest OK'}.items():
    o = subprocess.run(f'python -m {mod} --selftest', shell=True, capture_output=True, text=True)
    assert must in o.stdout, (mod, o.stdout[-800:], o.stderr[-800:])
    if mod.endswith('yesno_calib'):
        assert 'toán OK' in o.stdout, 'selftest bỏ qua phần toán (thiếu torch?) -> báo Claude'
        assert 'tiny Qwen2.5-VL OK' in o.stdout, 'selftest bỏ qua phép so với generate() -> báo Claude'
print('✅ selftest OK (2 script)')

## 3. Smoke test: cách đo v3 có tái hiện greedy W12 không? (≈ 2 phút)

Chấm 300 câu POPE-vi đầu tiên với SH, rồi so quyết định ở τ = 0 với greedy W12 trên cùng 300 câu. Đây chính là phép kiểm hợp lệ của luật, chạy sớm trên mẫu nhỏ để **không tốn ≈ 4,5 giờ GPU nếu cách đo vẫn sai**. Run smoke có `--limit 300`, nên `--analyze` sẽ tự từ chối nếu ai lỡ dùng nó cho luật.

*Kỳ vọng: dòng `ứng viên: {...} | token đầu: {...} | repetition_penalty (như generate) = 1.05`, `s/câu` ≈ 0,2–0,25, rồi `trùng greedy W12: ≥ 98% …` và `✅ … chạy tiếp`. Nếu `❌` → notebook DỪNG; gửi Claude output cell này.*

In [ ]:
from src.eval.yesno_calib import GREEDY_W12, RUN_TAG
SMOKE = f"W13_YN3_smoke300_{RUN_TAG['SH']}_popevi_bf16_L4_seed42"
if not os.path.exists(f'{EXP_ROOT}/{SMOKE}/logits.json'):
    run_cmd(f'python -u -m src.eval.yesno_calib --score --data_file {POPE_REPO} --run_name {SMOKE} '
            f'--drive_root {DRIVE_ROOT} --chunk 300 --limit 300 {OCR_ARGS} --adapter {AD["SH"]}')
m = json.load(open(f'{EXP_ROOT}/{SMOKE}/meta.json', encoding='utf-8'))
print('generation_config:', m['generation_config'])
print(f"s/câu: {m['sec_per_sample']} | VRAM đỉnh: {m.get('peak_vram_gb')} GB")
rc = run_cmd(f'python -m src.eval.yesno_calib --agree --run_dir {EXP_ROOT}/{SMOKE} --greedy_dir {EXP_ROOT}/{GREEDY_W12["SH"]}',
             check=False)
if rc != 0:
    raise SystemExit('⏸ DỪNG: cách đo v3 chưa tái hiện greedy -> gửi Claude output cell này (không chạy tiếp).')

## 4. Chấm 9 run: 3 run **bắt buộc** trước, 6 run chỉ mô tả sau

| Thứ tự | Run | Vai trò |
|---|---|---|
| 1–3 | SH calib · SH POPE-vi · B1-ocr POPE-vi | **bắt buộc** cho luật (≈ 13.700 câu ≈ 1 h) |
| 4–9 | SH t2 · B1-ocr calib · B1-ocr t2 · B3-full × 3 | chỉ mô tả (≈ 2 h) |

Mỗi câu = 1 forward 1 hàng, không sinh chữ (≈ 0,21–0,24 s/câu như v1).

Mỗi lô 500 câu ghi lên Drive ngay khi xong. **Bị ngắt thì Run all lại**: run đã xong tự bỏ qua, run dở chạy tiếp từ lô cuối. Muốn có kết quả sớm thì có thể dừng sau khi 3 run bắt buộc xong (thấy dòng `✅ 3 run bắt buộc xong`): bấm **Runtime → Interrupt execution**, rồi chạy riêng mục 5. `--analyze` vẫn phán quyết được và ghi các run mô tả còn thiếu.

*Kỳ vọng: mỗi run in `ứng viên … | repetition_penalty …`, các dòng lô, cuối cùng `== xong … câu | … s/câu`. Sau run thứ 3 có `✅ 3 run bắt buộc xong`.*

In [ ]:
from src.eval.yesno_calib import RUN_ORDER, REQUIRED_RUNS, logit_run_name
for i, (label, qset) in enumerate(RUN_ORDER, 1):
    run = logit_run_name(label, qset)
    if os.path.exists(f'{EXP_ROOT}/{run}/logits.json'):
        print(f'⏭ {i}. {label}/{qset}: đã có -> bỏ qua')
    else:
        cmd = (f'python -u -m src.eval.yesno_calib --score --data_file {Q_FILES[qset]} --run_name {run} '
               f'--drive_root {DRIVE_ROOT} --chunk {CHUNK} {OCR_ARGS}')
        if label in AD:
            cmd += f' --adapter {AD[label]}'
        run_cmd(cmd)
    m = json.load(open(f'{EXP_ROOT}/{run}/meta.json', encoding='utf-8'))
    print(f"✔ {i}. {label:6s} {qset:10s} {m['num_samples']:5d} câu | {m['sec_per_sample']} s/câu | VRAM {m.get('peak_vram_gb')} GB\n")
    if i == len(REQUIRED_RUNS):
        print('✅ 3 run bắt buộc xong -> có thể dừng ở đây và chạy mục 5 nếu cần kết quả sớm\n')

## 5. Phân tích + phán quyết theo luật R-YB v3

`src.eval.yesno_calib --analyze`:
1. **Kiểm hợp lệ**: đủ câu, đúng sha256, adapter, OCR, không phải run smoke, cùng repetition penalty. Quyết định τ = 0 phải trùng greedy W12 ≥ 98% (SH, B1-ocr).
2. **AUC** (chỉ mô tả), có cả bản không penalty để so.
3. **τ*** chọn trên calibration (balanced accuracy).
4. **Luật R-YB**: SH với τ* so với B1-ocr greedy trên POPE-vi, cộng yes-ratio.
5. **Chỉ mô tả**: các run còn lại (nếu đã chạy).

*Kỳ vọng: 6 bảng, dòng `➜ R-YB v3: **ĐẠT**` hoặc `**KHÔNG ĐẠT**`. File `yesno_calib_summary.json` + `.md` nằm trong `experiments/W13_yesno_calib_v3_L4_seed42/`.*

In [ ]:
os.makedirs(OUT, exist_ok=True)
run_cmd(f'python -m src.eval.yesno_calib --analyze --pope {POPE_REPO} --questions_dir {Q_DIR} '
        f'--exp_root {EXP_ROOT} --out {OUT}')

## 6. Tự trả máy L4 khi chạy xong

Kết quả đã nằm trên Drive. Output phía trên vẫn hiện trong tab sau khi trả máy. Nếu notebook dừng sớm vì lỗi thì máy **không** tự trả: đọc lỗi xong, tự bấm **Runtime → Disconnect and delete runtime**.

*Kỳ vọng: `✅ Xong toàn bộ -> tự trả máy …`.*

In [ ]:
AUTO_RELEASE = True
if AUTO_RELEASE:
    print('✅ Xong toàn bộ -> tự trả máy để ngừng tính CU (output phía trên vẫn giữ trong tab).')
    from google.colab import runtime
    runtime.unassign()
else:
    print('AUTO_RELEASE = False -> nhớ tự Disconnect and delete runtime khi xong.')

## 7. ⏸ DỪNG

Nhắn Claude **"xong"** + ảnh CU (Claude tự đọc kết quả trên Drive), hoặc gửi output **mục 3** và **mục 5**. Kết quả nằm trong `experiments/W13_YN3_*` và `W13_yesno_calib_v3_L4_seed42/` trên Drive.